In [1]:
import pymysql

In [14]:
import pymysql

conn = pymysql.connect(
    host='localhost',
    user='root',
    password='123456',
    db='projectdb',
    charset='utf8'
)

target_year_limit = 2042

try:
    cursor = conn.cursor()

    cursor.execute("SELECT * FROM poptbl WHERE year = 2024")
    row_2024 = cursor.fetchone()
    
    cursor.execute("DESC poptbl")
    col_names = [col[0] for col in cursor.fetchall()]
    
    current_data = dict(zip(col_names, row_2024))

    for year in range(2025, target_year_limit + 1):
        new_row_data = {'year': year}

        for i in range(1, 31):
            prev_age_col = str(i-1)
            curr_age_col = str(i)
            
            if prev_age_col in col_names and curr_age_col in col_names:
                new_row_data[curr_age_col] = current_data.get(prev_age_col, 0)

        quoted_cols = [f"`{c}`" for c in new_row_data.keys()]
        columns_sql = ', '.join(quoted_cols)
        placeholders = ', '.join(['%s'] * len(new_row_data))
        
        insert_sql = f"INSERT INTO poptbl ({columns_sql}) VALUES ({placeholders})"
        cursor.execute(insert_sql, list(new_row_data.values()))

        current_data = new_row_data.copy()

    conn.commit()
    print("성공")

except Exception as e:
    print("오류 발생:", e)
    conn.rollback()

finally:
    cursor.close()
    conn.close()

성공


In [15]:
conn = pymysql.connect(
    host='localhost',
    user='root',
    password='123456',
    db='projectdb',
    charset='utf8'
)

try:
    # 2. 18세부터 30세까지의 컬럼을 합산하는 SQL 구문 작성
    # 컬럼명이 숫자이므로 백틱(`)으로 감싸줍니다.
    age_columns = [f"`{age}`" for age in range(18, 31)]
    sum_query = " + ".join(age_columns)
    
    sql = f"SELECT year, ({sum_query}) as total_available FROM poptbl ORDER BY year"
    
    # 3. Pandas 데이터프레임으로 읽기
    df = pd.read_sql(sql, conn)

    # 4. 그래프 그리기
    plt.figure(figsize=(12, 6))
    plt.plot(df['year'], df['total_available'], marker='o', linestyle='-', color='b', linewidth=2)
    
    # 그래프 꾸미기
    plt.title('Annual Available Male Population (Age 18-30)', fontsize=15)
    plt.xlabel('Year', fontsize=12)
    plt.ylabel('Population Count', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.xticks(df['year'], rotation=45) # 연도가 잘 보이게 회전
    
    # 천 단위 콤마 표시
    current_values = plt.gca().get_yticks()
    plt.gca().set_yticklabels(['{:,.0f}'.format(x) for x in current_values])

    plt.tight_layout()
    plt.show()

    # 데이터 수치 확인용 출력
    print(df)

except Exception as e:
    print("오류 발생:", e)

finally:
    conn.close()

오류 발생: name 'pd' is not defined
